[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heitorramos/icd/blob/main/exemplos/19-verossimilhanca-regressao/notebook-colab.ipynb)


In [ ]:
# Preparação automática para execução no Google Colab.
# Fora do Colab, esta célula não altera o diretório de trabalho.
try:
    import google.colab  # type: ignore
except ImportError:
    pass
else:
    import os
    import subprocess
    from pathlib import Path

    repository = Path("/content/icd")
    if not repository.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/heitorramos/icd.git", str(repository)
        ], check=True)
    os.chdir(repository / "exemplos/19-verossimilhanca-regressao")
    print("Material preparado em:", Path.cwd())


# Verossimilhança gaussiana e regressão linear

Material de apoio — Aula 19

## Objetivos

Este guia formula a regressão como modelo probabilístico, deriva sua
log-verossimilhança e mostra por que máxima verossimilhança gaussiana e
mínimos quadrados estimam os mesmos coeficientes. A discussão final
esclarece o papel real da normalidade.

## Como estudar este capítulo

Na Aula 17, a reta foi escolhida por minimizar a soma dos resíduos ao
quadrado. Agora perguntamos: **qual modelo probabilístico conduz
exatamente ao mesmo critério?** A resposta aparece quando supomos que,
para cada valor dos preditores, a resposta oscila ao redor da reta
segundo uma distribuição Normal com variância constante.

O capítulo separa duas camadas que muitas vezes são misturadas. A
primeira é a **média condicional**, descrita pela reta. A segunda é a
**distribuição das observações ao redor dessa média**, descrita pelo
termo de erro. A normalidade pertence à segunda camada. Ela não é
necessária para desenhar a reta de mínimos quadrados, mas sustenta a
forma específica da verossimilhança e os procedimentos inferenciais
usuais em amostras pequenas.

Ao acompanhar as equações, identifique sempre o papel de cada termo:
$y_i$ é a resposta observada, $x_i^T\beta$ é a média prevista,
$y_i-x_i^T\beta$ é o resíduo correspondente e $\sigma^2$ controla a
dispersão ao redor da reta.

## Base de dados de apoio

Continuamos com [Medical Insurance
Cost](https://www.kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset).
A resposta é `charges`, o preditor é `age` e `smoker` será usado no
diagnóstico.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
df = pd.read_csv(Path("../16-correlacao/data/insurance.csv"))
df[["age", "charges", "smoker"]].head()

> **Interpretação**
>
> Cada linha é uma pessoa. A regressão simples descreve despesas médias
> por idade; ela não incorpora todas as características que distinguem
> indivíduos com a mesma idade.

## Modelo para a média condicional

$$Y_i=\beta_0+\beta_1x_i+\varepsilon_i,
\qquad E[\varepsilon_i\mid X_i=x_i]=0.$$

$Y_i$ é a resposta, $x_i$ o preditor, $\beta_0$ o intercepto, $\beta_1$
a inclinação e $\varepsilon_i$ o erro populacional. A condição de média
zero implica

$$E[Y_i\mid X_i=x_i]=\beta_0+\beta_1x_i.$$

## Acrescentando uma distribuição

Para construir a verossimilhança gaussiana, assumimos

$$\varepsilon_i\overset{iid}{\sim}N(0,\sigma^2).$$

Equivalentemente,

$$Y_i\mid X_i=x_i\sim N(\beta_0+\beta_1x_i,\sigma^2).$$

A média depende de $x_i$ e a variância condicional é constante.

## Densidade condicional

$$
f(y_i\mid x_i;\beta_0,\beta_1,\sigma^2)
=\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\left[-\frac{(y_i-\beta_0-\beta_1x_i)^2}{2\sigma^2}\right].
$$

O termo $y_i-\beta_0-\beta_1x_i$ é o erro vertical relativo a uma reta
candidata. Erros maiores recebem densidade menor.

## Verossimilhança conjunta

Sob independência condicional,

$$
L(\beta_0,\beta_1,\sigma^2)
=\prod_{i=1}^n f(y_i\mid x_i;\beta_0,\beta_1,\sigma^2).
$$

Tomando log:

$$
\ell=-\frac n2\log(2\pi)-\frac n2\log(\sigma^2)
-\frac{1}{2\sigma^2}\sum_i(y_i-\beta_0-\beta_1x_i)^2.
$$

## Equivalência com mínimos quadrados

Para $\sigma^2>0$ fixo, os dois primeiros termos não dependem dos
coeficientes. Portanto,

$$
\arg\max_{\beta_0,\beta_1}\ell
=\arg\min_{\beta_0,\beta_1}
\sum_i(y_i-\beta_0-\beta_1x_i)^2.
$$

> **Interpretação**
>
> OLS e máxima verossimilhança gaussiana não são dois algoritmos que
> coincidiram por acaso. O log negativo da densidade normal é uma
> constante mais o erro quadrático escalado.

## Cálculo manual dos coeficientes

In [ ]:
x = df["age"].to_numpy(float)
y = df["charges"].to_numpy(float)
xbar, ybar = x.mean(), y.mean()
b1 = np.sum((x-xbar)*(y-ybar))/np.sum((x-xbar)**2)
b0 = ybar-b1*xbar
yhat = b0+b1*x
resid = y-yhat
pd.Series({"beta0": b0, "beta1": b1}).round(4)

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=.4)
order = np.argsort(x)
plt.plot(x[order], yhat[order], color="darkorange", linewidth=3)
plt.show()

> **Interpretação**
>
> Um ano adicional de idade está associado a aproximadamente US\$ 257,72
> na despesa média prevista. O grande espalhamento vertical mostra que
> idade isolada produz previsões individuais frágeis.

## Verificação numérica da equivalência

In [ ]:
sigma2 = np.mean(resid**2)

def sse(beta0, beta1):
    return np.sum((y-beta0-beta1*x)**2)

def loglik(beta0, beta1, sigma2):
    return (-len(y)/2*np.log(2*np.pi)
            -len(y)/2*np.log(sigma2)
            -sse(beta0, beta1)/(2*sigma2))

candidates = pd.DataFrame({
    "modelo": ["inclinação menor", "OLS/MLE", "inclinação maior"],
    "beta1": [.5*b1, b1, 1.5*b1],
})
candidates["beta0"] = ybar-candidates.beta1*xbar
candidates["SSE"] = [sse(a,b) for a,b in zip(candidates.beta0,candidates.beta1)]
candidates["logLik"] = [loglik(a,b,sigma2) for a,b in zip(candidates.beta0,candidates.beta1)]
candidates.round(2)

> **Interpretação**
>
> O candidato com menor SSE é exatamente o de maior log-verossimilhança.
> Alterar a inclinação enquanto a reta passa pelo centro dos dados piora
> ambos os critérios em direções opostas.

## Estimação da variância

Maximizando em $\sigma^2$:

$$
\widehat\sigma^2_{MV}=\frac1n\sum_i e_i^2.
$$

In [ ]:
sigma2_mle = np.mean(resid**2)
sigma2_unbiased = np.sum(resid**2)/(len(y)-2)
pd.Series({
    "sigma_MLE": np.sqrt(sigma2_mle),
    "sigma_corrigido": np.sqrt(sigma2_unbiased),
}).round(2)

> **Interpretação**
>
> Dividir por $n$ maximiza a verossimilhança. Dividir por $n-2$ corrige
> o viés causado pela estimação de intercepto e inclinação. “MLE” e “não
> viesado” são propriedades diferentes.

## Os resíduos precisam ser normais?

Não para calcular OLS. A minimização da SSE é uma operação algébrica que
não exige distribuição. Para que OLS estime a média condicional sem viés
sistemático, a condição essencial é $E[\varepsilon\mid X]=0$.

Normalidade é usada para declarar a verossimilhança gaussiana correta e
obter certas distribuições exatas de estatísticas em amostras finitas.

## Erros e resíduos

- $\varepsilon_i$ é o erro populacional não observado;
- $e_i=y_i-\widehat y_i$ é o resíduo após estimar a reta;
- resíduos somam zero quando há intercepto;
- resíduos não são independentes entre si;
- sua distribuição serve como aproximação diagnóstica à distribuição dos
  erros.

## Diagnóstico da base

In [ ]:
diagnostics = pd.DataFrame({
    "ajustado": yhat,
    "resíduo": resid,
    "fumante": df["smoker"],
})
diagnostics.groupby("fumante")["resíduo"].agg(["count", "mean", "std"]).round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(resid, kde=True, ax=axes[0])
sns.scatterplot(data=diagnostics, x="ajustado", y="resíduo", hue="fumante", alpha=.5, ax=axes[1])
axes[1].axhline(0, color="black", linestyle="--")
plt.tight_layout(); plt.show()

> **Interpretação**
>
> Os resíduos são assimétricos e o tabagismo separa dois patamares. O
> problema mais evidente não é apenas “falta de normalidade”: a função
> média está mal especificada porque omite uma variável fortemente
> associada às despesas.

## Consequências de violações

| hipótese | se falhar |
|------------------------------------|------------------------------------|
| média linear correta | coeficientes podem resumir uma forma inadequada |
| $E[\varepsilon\mid X]=0$ | associação pode estar enviesada por estrutura omitida |
| variância constante | erro-padrão clássico pode estar errado |
| independência | produto da verossimilhança e incerteza usual falham |
| normalidade | inferência exata gaussiana e previsão normal podem falhar |

## Outras distribuições, outras perdas

Para a Normal,

$$-\log f(e)=C+e^2/(2\sigma^2).$$

Por isso aparece a perda quadrática. Erros Laplace produzem perda
absoluta; Bernoulli produz log-loss. Escolher uma distribuição é também
escolher como discrepâncias serão penalizadas.

## Da reta à verossimilhança: o caminho completo

Nesta aula, acrescentamos uma camada probabilística à regressão da Aula
17:

1.  **A reta fornece a média.** Para cada idade $x_i$, calculamos
    $\mu_i=\beta_0+\beta_1x_i$.
2.  **Os indivíduos não ficam exatamente sobre a média.** Representamos
    a diferença por um erro $\varepsilon_i$.
3.  **Escolhemos uma distribuição para esses erros.** A Normal descreve
    erros simétricos em torno de zero, com escala controlada por
    $\sigma$.
4.  **Cada reta candidata gera densidades diferentes para os dados.**
    Uma reta distante dos pontos atribui densidade pequena às
    observações.
5.  **Multiplicamos as contribuições.** Sob independência, obtemos a
    verossimilhança conjunta.
6.  **Maximizamos.** Com erros normais e variância constante, isso
    equivale a minimizar a soma dos quadrados dos resíduos.

Essa equivalência explica por que mínimos quadrados reaparece. O
quadrado está no expoente da densidade Normal; ao tomar o log, ele se
transforma justamente na soma de quadrados.

## Os resíduos precisam parecer perfeitamente normais?

Não. A reta pode ser calculada mesmo quando a distribuição dos resíduos
não é Normal. A normalidade é mais importante para intervalos e testes
clássicos em amostras pequenas.

Na prática, olhamos primeiro para problemas mais estruturais:

- tendência curva nos resíduos: a média não foi bem representada;
- formato de funil: a variabilidade muda com o valor previsto;
- grupos separados: pode faltar uma variável importante;
- poucos resíduos extremos: algumas observações podem dominar o ajuste;
- dependência temporal ou por pessoa: a informação efetiva é menor do
  que o número de linhas sugere.

O QQ-plot é uma parte desse diagnóstico, não um teste que aprova ou
reprova sozinho toda a regressão.

> **Ideia central**
>
> Mínimos quadrados diz como escolher a reta. A hipótese Normal explica
> como transformar a distância dos pontos à reta em uma verossimilhança
> e em procedimentos de inferência.

## Questões de revisão

1.  Derive a log-verossimilhança gaussiana a partir do produto.
2.  Identifique os termos constantes em $\beta_0,\beta_1$.
3.  Por que maximizar $-\operatorname{SSE}/(2\sigma^2)$ equivale a
    minimizar SSE?
4.  Diferencie erro normal, resíduo aparentemente normal e média
    condicional correta.
5.  Por que o padrão por tabagismo sugere revisar a média, não apenas a
    variância?

## Bibliografia

- James et al., *An Introduction to Statistical Learning*, capítulo 3.
- Wasserman, *All of Statistics*, MLE e regressão.
- Faraway, *Linear Models with R*, capítulos 2–4.
- Gelman, Hill e Vehtari, *Regression and Other Stories*, capítulos
  6–11.
- Fox, *Applied Regression Analysis and Generalized Linear Models*,
  diagnóstico.